# Buổi 19 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `gian_doan.py`; các ô tự dùng bản mới.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Dữ liệu và phân loại (mục 4.1)

In [ ]:
%matplotlib inline
import warnings

import gian_doan as gd
import numpy as np

warnings.simplefilter("ignore")
df = gd.doc_car_parts()
Y = gd.ma_tran(df)
print("số mã:", len(Y), "| số tháng:", Y.shape[1], "| tỷ lệ ô bằng 0:", round(float((Y == 0).to_numpy().mean()), 3))
hoc = Y.iloc[:, :gd.SO_THANG - gd.SO_KY_CHAM - 1]      # 38 tháng trước cutoff đầu tiên
print(gd.bang_phan_loai(hoc)["loại"].value_counts().to_string())
print(gd.phan_loai(np.array([0, 3, 0, 0, 5, 0, 4, 0])))

## Bước 2 — Croston và TSB tự viết (mục 4.3–4.4)

Sửa `tsb` rồi chạy lại ô này.

In [ ]:
from statsforecast.models import TSB, CrostonClassic, CrostonSBA

y = np.array([0, 0, 3, 0, 0, 0, 2, 0, 4, 0, 0, 0], float)
print("Croston tự viết", round(gd.croston(y), 3), "| thư viện", round(float(CrostonClassic().forecast(y, 1)["mean"][0]), 3))
print("SBA     tự viết", round(gd.croston(y, sba=True), 3), "| thư viện", round(float(CrostonSBA().forecast(y, 1)["mean"][0]), 3))
print("TSB     tự viết", round(gd.tsb(y), 3), "| thư viện", round(float(TSB(0.1, 0.1).forecast(y, 1)["mean"][0]), 3))

v = Y.loc["T1002"].to_numpy()                         # ngừng bán sau tháng 17
print("tháng 50 — Croston", round(gd.croston(v), 3), "| TSB", round(gd.tsb(v), 3))

## Bước 3 — Backtest và bảng chỉ số (mục 4.2)

Lần đầu khoảng 15 giây; kết quả lưu vào `lab/du-lieu/cache/`. Sửa `danh_gia` rồi chạy lại ô này.

In [ ]:
kq = gd.backtest_car_parts(df, luu=True)
bang = gd.danh_gia(kq, Y)
print(bang.round(3).to_string())

## Bước 4 — Mô phỏng tồn kho (mục 4.6)

Sửa `chon_mo_hinh` rồi chạy lại ô này.

In [ ]:
print(gd.mo_phong_ton_kho(np.array([[3, 0, 2]]), np.array([[2, 2, 2]])))   # ví dụ tay: chi phí 11, đáp ứng 0,8
tong = bang.join(gd.chi_phi_ton_kho(kq, Y))
print(tong.round(3).sort_values("chi phí mỗi mã").to_string())
print("chọn:", gd.chon_mo_hinh(tong))

## Bước 5 — Kiểm tra

Trong terminal ở thư mục `lab/`: `python lab.py check` — phải xanh 8/8.